# FRG frozen-flow debugger notebook

This notebook does **not** modify the original source files.

It is designed to answer only one question:

**Why are one-loop kernels nonzero in sanity checks, but `FRGFlowSolver` still produces `rhs_norm = 0` and a frozen flow?**


In [1]:
import json
import traceback
import numpy as np
import pandas as pd

from frg_flow import BareVertexFromInteraction, FRGFlowSolver
from frg_kernel import FlowConfig
from debug_frg_flow import DebugFRGFlowSolver, compare_kernel_interfaces


## 1. Recreate the same model / patchsets / interaction as your existing benchmark

Use the **same setup** you already used in `benchmark3_frg_physics_sanity_checks.ipynb`.

This cell is intentionally left as a template so you can paste your own working setup.


In [ ]:
# TODO: paste the exact setup from your existing benchmark notebook here.
# Expected objects after this cell:
#
# patchsets      : dict-like map, e.g. {"up": patchset_up, "dn": patchset_dn}
# interaction    : interaction object used by BareVertexFromInteraction
# diagnoser      : optional; can be None
#
# Example placeholder:
#
# from noninteracting import ...
# from patching import ...
# from interaction import ...
# ...
# patchsets = {"up": patchset_up, "dn": patchset_dn}
# interaction = your_interaction_object
# diagnoser = None
#
raise RuntimeError("Paste your benchmark setup into this cell before running.")


In [5]:
import numpy as np
import matplotlib.pyplot as plt

from noninteracting import KagomeKaneMeleSOC, KagomeStaggerFlux, KagomeNagaosa
from patching import FSPatcher
from interaction import BareExtendedHubbard
from channels import ChannelDecomposer
from frg_kernel import (
    FlowConfig,
    compute_pp_kernel,
    compute_ph_kernel,
    compute_phc_kernel,
)
from kagome_order_diagnosis import KagomeOrderDiagnoser

# Optional: only if your current Module 4 solver exists and works
try:
    from frg_flow import FRGFlowSolver, BareVertexFromInteraction
    HAS_FLOW_SOLVER = True
except Exception:
    HAS_FLOW_SOLVER = False
    print("frg_flow.py not available or not importable; flow tests will be notebook-design only.")


In [7]:
def build_spin_patchsets(model, mu, band_index=1, Npatch=12, grid_size=220):
    patch_up = FSPatcher(
        model,
        band_index=band_index,
        mu=mu,
        Npatch=Npatch,
        grid_size=grid_size,
        orbital_slice=slice(0, 3),
        gauge_fix="parallel_transport",
    ).build()

    patch_dn = FSPatcher(
        model,
        band_index=band_index,
        mu=mu,
        Npatch=Npatch,
        grid_size=grid_size,
        orbital_slice=slice(3, 6),
        gauge_fix="parallel_transport",
    ).build()

    return {"up": patch_up, "dn": patch_dn}


def build_reference_model_and_patchsets(
    *,
    model_name="kane_mele",
    parameters=None,
    filling=0.50,
    Npatch=12,
    grid_size=220,
    band_index=1,
):
    if model_name == "kane_mele":
        if parameters is None:
            parameters = {"t": 1.0, "l1": 0.0, "l2": 0.0}
        model = KagomeKaneMeleSOC(parameters=parameters, spin=True, B=0.0)
    elif model_name == "stagger_flux":
        if parameters is None:
            parameters = {"t": 1.0, "phi": 0.0}
        model = KagomeStaggerFlux(parameters=parameters, spin=True, B=0.0)
    elif model_name == "nagaosa":
        if parameters is None:
            parameters = {"t": 1.0, "phi": 0.0}
        model = KagomeNagaosa(parameters=parameters, spin=True, B=0.0)
    else:
        raise ValueError("Unknown model_name")

    mu = model.EF_from_filling(filling)
    patchsets = build_spin_patchsets(model, mu, band_index=band_index, Npatch=Npatch, grid_size=grid_size)
    return model, mu, patchsets


def make_interaction(model, U, V):
    return BareExtendedHubbard.from_kagome_model(model, U=U, V=V)


def block_tensor_norm(T):
    arr = np.asarray(T)
    return float(np.max(np.abs(arr)))


def kernel_norm(kernel):
    return float(np.max(np.abs(kernel.matrix)))


def relative_diff(a, b, floor=1e-14):
    a = np.asarray(a)
    b = np.asarray(b)
    denom = max(float(np.max(np.abs(a))), float(np.max(np.abs(b))), floor)
    return float(np.max(np.abs(a - b)) / denom)


def build_gamma_tensors(interaction, patchsets):
    spin_blocks = [
        ("up", "up", "up", "up"),
        ("dn", "dn", "dn", "dn"),
        ("up", "dn", "up", "dn"),
        ("up", "dn", "dn", "up"),
        ("dn", "up", "up", "dn"),
        ("dn", "up", "dn", "up"),
    ]
    out = {}
    for key in spin_blocks:
        s1, s2, s3, s4 = key
        out[key] = interaction.patch_tensor(
            patchsets, s1, s2, s3, s4,
            antisym=True,
            enforce_momentum=False,
        )
    return out


def pick_test_Qs(patchsets, n_pick=6):
    patch_up = patchsets["up"]
    ks = patch_up.patch_k
    idx = np.linspace(0, len(ks)-1, num=min(n_pick, len(ks)), dtype=int)
    return [ks[i] for i in idx]


def compute_basic_channel_set(gamma, patchsets, Q, T=0.20, nfreq=64):
    cfg = FlowConfig(temperature=T, nfreq=nfreq, include_explicit_T_prefactor=True)

    out = {}
    try:
        out["pp_ud_ud"] = compute_pp_kernel(
            gamma, patchsets, Q,
            incoming_spins=("up", "dn"),
            outgoing_spins=("up", "dn"),
            config=cfg,
        )
    except Exception as e:
        out["pp_ud_ud_error"] = e

    try:
        out["ph_uu"] = compute_ph_kernel(
            gamma, patchsets, Q,
            incoming_spins=("up", "up"),
            outgoing_spins=("up", "up"),
            config=cfg,
        )
    except Exception as e:
        out["ph_uu_error"] = e

    try:
        out["ph_dd"] = compute_ph_kernel(
            gamma, patchsets, Q,
            incoming_spins=("dn", "dn"),
            outgoing_spins=("dn", "dn"),
            config=cfg,
        )
    except Exception as e:
        out["ph_dd_error"] = e

    try:
        out["phc_uu"] = compute_phc_kernel(
            gamma, patchsets, Q,
            incoming_spins=("up", "up"),
            outgoing_spins=("up", "up"),
            config=cfg,
        )
    except Exception as e:
        out["phc_uu_error"] = e

    try:
        out["phc_dd"] = compute_phc_kernel(
            gamma, patchsets, Q,
            incoming_spins=("dn", "dn"),
            outgoing_spins=("dn", "dn"),
            config=cfg,
        )
    except Exception as e:
        out["phc_dd_error"] = e

    return out


In [11]:
if HAS_FLOW_SOLVER:
    model, mu, patchsets = build_reference_model_and_patchsets(
        model_name="kane_mele",
        parameters={"t": 1.0, "l1": 0.0, "l2": 0.0},
        filling=0.50,
        Npatch=10,
    )
    interaction = make_interaction(model, U=0.10, V=0.0)
    bare_gamma = BareVertexFromInteraction(interaction, patchsets)
    diagnoser = KagomeOrderDiagnoser(patchsets_by_spin=patchsets)

    solver = FRGFlowSolver(
        patchsets=patchsets,
        bare_gamma=bare_gamma,
        diagnoser=diagnoser,
        T_start=0.40,
        T_stop=0.10,
        n_steps=6,
        nfreq=32,
        diagnose_every=1,
    )
#     history = solver.run()
#     for h in history:
#         print(h.summary_dict())
# else:
#     print("Skip: FRGFlowSolver not importable.")


## 2. Build the debug solver with small settings first

In [13]:
solver = DebugFRGFlowSolver(
    patchsets=patchsets,
    bare_gamma=BareVertexFromInteraction(interaction, patchsets),
    diagnoser=diagnoser if 'diagnoser' in globals() else None,
    T_start=0.20,
    T_stop=0.10,
    n_steps=3,
    nfreq=64,
    include_explicit_T_prefactor=True,
    diagnose_every=1,
)
print("n_pp_Q =", len(solver.pp_grid.q_list))
print("n_ph_Q =", len(solver.phd_grid.q_list))
print("n_phc_Q =", len(solver.phc_grid.q_list))
print("spin_blocks =", solver.spin_blocks)


n_pp_Q = 53
n_ph_Q = 51
n_phc_Q = 51
spin_blocks = [('up', 'up', 'up', 'up'), ('dn', 'dn', 'dn', 'dn'), ('up', 'dn', 'up', 'dn'), ('up', 'dn', 'dn', 'up'), ('dn', 'up', 'up', 'dn'), ('dn', 'up', 'dn', 'up')]


## 3. Run detailed RHS debug once

In [15]:
T0 = float(solver.state.T)
rhs_pp, rhs_phd, rhs_phc, attempts, summary = solver.compute_channel_rhs_debug(T0, raise_on_error=False)

print(json.dumps(summary, indent=2))

attempt_df = pd.DataFrame([a.to_dict() for a in attempts])
attempt_df.head(20)


{
  "total": 930,
  "success": 0,
  "failure": 930,
  "by_channel": {
    "pp": {
      "success": 0,
      "failure": 318,
      "nonzero_success": 0,
      "max_abs": 0.0
    },
    "phd": {
      "success": 0,
      "failure": 306,
      "nonzero_success": 0,
      "max_abs": 0.0
    },
    "phc": {
      "success": 0,
      "failure": 306,
      "nonzero_success": 0,
      "max_abs": 0.0
    }
  },
  "top_errors": {
    "pp:TypeError:'float' object is not subscriptable": 318,
    "phc:TypeError:'float' object is not subscriptable": 306,
    "phd:TypeError:'float' object is not subscriptable": 306
  },
  "nonzero_success_count": 0,
  "max_success_abs": 0.0
}


,channel,iq,Q,spin_block,success,max_abs,exc_type,exc_message
0,pp,0,"[0.0, 0.0]","[up, up, up, up]",False,None,TypeError,'float' object is not subscriptable
1,pp,0,"[0.0, 0.0]","[dn, dn, dn, dn]",False,None,TypeError,'float' object is not subscriptable
2,pp,0,"[0.0, 0.0]","[up, dn, up, dn]",False,None,TypeError,'float' object is not subscriptable
3,pp,0,"[0.0, 0.0]","[up, dn, dn, up]",False,None,TypeError,'float' object is not subscriptable
4,pp,0,"[0.0, 0.0]","[dn, up, up, dn]",False,None,TypeError,'float' object is not subscriptable
5,pp,0,"[0.0, 0.0]","[dn, up, dn, up]",False,None,TypeError,'float' object is not subscriptable
6,pp,1,"[-0.2868137543768711, -2.7035848267983913]","[up, up, up, up]",False,None,TypeError,'float' object is not subscriptable
7,pp,1,"[-0.2868137543768711, -2.7035848267983913]","[dn, dn, dn, dn]",False,None,TypeError,'float' object is not subscriptable
8,pp,1,"[-0.2868137543768711, -2.7035848267983913]","[up, dn, up, dn]",False,None,TypeError,'float' object is not subscriptable
9,pp,1,"[-0.2868137543768711, -2.7035848267983913]","[up, dn, dn, up]",False,None,TypeError,'float' object is not subscriptable


## 4. Show only failed kernel builds

In [16]:
fail_df = attempt_df[attempt_df["success"] == False].copy()
fail_df.sort_values(["channel", "iq"]).head(50)


,channel,iq,Q,spin_block,success,max_abs,exc_type,exc_message
624,phc,0,"[0.0, 0.0]","[up, up, up, up]",False,None,TypeError,'float' object is not subscriptable
625,phc,0,"[0.0, 0.0]","[dn, dn, dn, dn]",False,None,TypeError,'float' object is not subscriptable
626,phc,0,"[0.0, 0.0]","[up, dn, up, dn]",False,None,TypeError,'float' object is not subscriptable
627,phc,0,"[0.0, 0.0]","[up, dn, dn, up]",False,None,TypeError,'float' object is not subscriptable
628,phc,0,"[0.0, 0.0]","[dn, up, up, dn]",False,None,TypeError,'float' object is not subscriptable
629,phc,0,"[0.0, 0.0]","[dn, up, dn, up]",False,None,TypeError,'float' object is not subscriptable
630,phc,1,"[0.8078781669325225, 0.1955046238754563]","[up, up, up, up]",False,None,TypeError,'float' object is not subscriptable
631,phc,1,"[0.8078781669325225, 0.1955046238754563]","[dn, dn, dn, dn]",False,None,TypeError,'float' object is not subscriptable
632,phc,1,"[0.8078781669325225, 0.1955046238754563]","[up, dn, up, dn]",False,None,TypeError,'float' object is not subscriptable
633,phc,1,"[0.8078781669325225, 0.1955046238754563]","[up, dn, dn, up]",False,None,TypeError,'float' object is not subscriptable


## 5. Count failures by exception type

In [18]:
if len(fail_df):
    display(fail_df.groupby(["channel", "exc_type", "exc_message"]).size().reset_index(name="count").sort_values("count", ascending=False))
else:
    print("No failures found.")


,channel,exc_type,exc_message,count
2,pp,TypeError,'float' object is not subscriptable,318
0,phc,TypeError,'float' object is not subscriptable,306
1,phd,TypeError,'float' object is not subscriptable,306


## 6. Check whether successful kernels are actually nonzero

In [ ]:
succ_df = attempt_df[attempt_df["success"] == True].copy()
display(succ_df.groupby("channel")["max_abs"].agg(["count", "max", "min", "mean"]))
display(succ_df[succ_df["max_abs"] > 1e-14].groupby("channel")["max_abs"].agg(["count", "max", "min", "mean"]))


## 7. Compare bare callable vertex vs flow gamma accessor on one representative Q

In [ ]:
cfg = FlowConfig(temperature=T0, nfreq=64, include_explicit_T_prefactor=True)

Q_test = solver.phd_grid.q_list[0]
cmp_result = compare_kernel_interfaces(
    patchsets=patchsets,
    interaction=interaction,
    Q=Q_test,
    config=cfg,
    spin_block=("up", "dn", "up", "dn"),
    solver=solver,
)
print(json.dumps(cmp_result, indent=2))


Interpretation:

- If `bare_success = False`, the issue is already in `BareVertexFromInteraction` / interaction interface.
- If `bare_success = True` but `flow_success = False`, the issue is in `solver.gamma_accessor()`.
- If both succeed but `flow_max_abs = 0`, then the reconstructed flowing vertex path is numerically zero.


## 8. Perform one debug step and see whether channel norms change

In [ ]:
step_info = solver.debug_single_step()
print(json.dumps({k: v for k, v in step_info.items() if k != "attempts"}, indent=2))


## 9. Optional: hard fail on first exception to get the full traceback

In [24]:
# Uncomment this if you want the notebook to stop exactly at the first failing kernel.

rhs_pp, rhs_phd, rhs_phc, attempts, summary = solver.compute_channel_rhs_debug(T0, raise_on_error=True)


TypeError: 'float' object is not subscriptable

In [26]:
gamma = solver.gamma_accessor()

# 随便取一个点
val = gamma(0, "up", 0, "dn", 0, "up", 0, "dn")
print(val)

(0.06956668720152745+0j)


In [28]:
from frg_flow import BareVertexFromInteraction

gamma_bare = BareVertexFromInteraction(interaction, patchsets)

val_bare = gamma_bare(0, "up", 0, "dn", 0, "up", 0, "dn")
print(val_bare)

(0.06956668720152745+0j)


In [30]:
# 检查 flow state 实际初始化了哪些 spin block
existing_blocks = set((k[0], k[1], k[2], k[3]) for k in solver.state.pp_corr.keys())
print("existing blocks:")
for blk in sorted(existing_blocks):
    print(blk)

# 检查 pp kernel 在一个 Q 上可能会请求哪些 spin block
requested_blocks = set()

s1, s2, s3, s4 = ("up", "dn", "up", "dn")   # 你可以先用这个外部块
for sa in ("up", "dn"):
    for sb in ("up", "dn"):
        requested_blocks.add((s1, s2, sa, sb))
        requested_blocks.add((sa, sb, s3, s4))

print("\nrequested blocks in pp kernel:")
for blk in sorted(requested_blocks):
    print(blk)

missing = requested_blocks - existing_blocks
print("\nmissing blocks:")
for blk in sorted(missing):
    print(blk)

existing blocks:
('dn', 'dn', 'dn', 'dn')
('dn', 'up', 'dn', 'up')
('dn', 'up', 'up', 'dn')
('up', 'dn', 'dn', 'up')
('up', 'dn', 'up', 'dn')
('up', 'up', 'up', 'up')

requested blocks in pp kernel:
('dn', 'dn', 'up', 'dn')
('dn', 'up', 'up', 'dn')
('up', 'dn', 'dn', 'dn')
('up', 'dn', 'dn', 'up')
('up', 'dn', 'up', 'dn')
('up', 'dn', 'up', 'up')
('up', 'up', 'up', 'dn')

missing blocks:
('dn', 'dn', 'up', 'dn')
('up', 'dn', 'dn', 'dn')
('up', 'dn', 'up', 'up')
('up', 'up', 'up', 'dn')


## 10. What file is most likely wrong?

This debugger is aimed mainly at **`frg_flow.py`**, because the original `compute_channel_rhs()` silently swallows every kernel exception.

But depending on what the notebook shows, the root cause may instead be one of:
- `interaction.py`
- `frg_kernel.py`
- `patching.py`

Use the failure table and the interface-comparison cell to decide.
